# Korrelations- und Verteilungsanalyse: Körpergröße, Schuhgröße und Gewicht

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_04/korrelationsanalyse_beispiel.ipynb)

**Lernziele:**
- Normalverteilung prüfen und fitten
- Scatterplot und linearer Fit
- Korrelationen berechnen und interpretieren
- Korrelation ist nicht Kausalität
- Optionale Vertiefung: statistische Tests (Shapiro-Wilk, p-Wert) und ihre Tücken

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def p_text(p):
    # Sehr kleine p-Werte nicht als 0.000 anzeigen: p ist nie exakt 0
    return 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'

## 1. Datenerstellung

Die Daten sind **Beispieldaten** von 21 bzw. 19 Personen (keine echte Studie).

In [ ]:
# Drei Listen mit je 21 Werten (eine Person pro Position)
#koerpergroesse = [188, 194, 178, 179, 165, 163, 172, 183, 181, 195, 186, 196, 179, 190, 187, 165, 190, 169, 183, 177, 170]
#gewicht = [108, 91, 60, 84, 70, 58, 72, 80, 70, 90, 93, 96, 95, 83, 99, 55, 85, 62, 84, 77, 70]
#schuhgroesse = [45, 43, 42, 44, 39, 39, 42, 43, 43, 47, 45, 47, 46, 46, 44, 41, 47, 39, 44, 42, 40]

# Drei Listen mit je 19 Werten (eine Person pro Position)
koerpergroesse = [188, 186, 183, 178, 180, 167, 160, 181, 180, 176, 182, 183, 179, 188, 177, 183, 179, 175, 180]
gewicht = [108, 80, 75, 68, 73, 58, 56, 82, 87, 54, 67, 84, 78, 69, 83, 84, 105, 72, 76]
schuhgroesse = [45, 44, 45, 43, 42, 37, 38, 43, 42, 40, 43, 43, 44, 44, 45, 43, 45, 42, 43]

df = pd.DataFrame({
    'Körpergröße [cm]': koerpergroesse,
    'Schuhgröße [EU]': schuhgroesse,
    'Gewicht [kg]': gewicht
})
print("=== DATEN ===")
print(df.head(10))
print('Deskriptive Statistik:')
print(df.describe().round(2))  # std = Standardabweichung (pandas teilt durch n-1)

## 2. Histogramm mit Normalverteilungs-Fit

Wir wählen die **Körpergröße** und überprüfen, ob sie normalverteilt aussieht.

Kern dieses Abschnitts: Histogramm und Normalverteilungs-Kurve. Der **Shapiro-Wilk-Test** ist eine *optionale Vertiefung* (wird in der Vorlesung später behandelt). Du kannst ihn zunächst überspringen.

In [ ]:
variable = 'Körpergröße [cm]'
data = df[variable].values

# Parameter der Normalverteilung schätzen (Mittelwert μ, Standardabweichung σ)
# Hinweis: norm.fit teilt bei σ durch n, pandas (describe) durch n-1 -> kleiner Unterschied
mu, sigma = stats.norm.fit(data)

fig, ax = plt.subplots(figsize=(10, 6))
n_bins = 7  # bei nur 19 Werten sind wenige Klassen übersichtlicher
ax.hist(data, bins=n_bins, density=True, alpha=0.7, color='skyblue', edgecolor='black')
x_fit = np.linspace(data.min() - 5, data.max() + 5, 100)
y_fit = stats.norm.pdf(x_fit, mu, sigma)
ax.plot(x_fit, y_fit, 'r-', linewidth=3, label=f'Normalverteilung\nμ = {mu:.1f}, σ = {sigma:.1f}')
ax.set_xlabel(variable)
ax.set_ylabel('Dichte')
ax.set_title(f'Histogramm: {variable} mit Normalverteilungs-Fit')
ax.legend()
ax.grid(True, alpha=0.3)

# Optionale Vertiefung: Shapiro-Wilk-Test
shapiro_stat, shapiro_p = stats.shapiro(data)
ax.text(0.02, 0.98, f'Shapiro-Wilk Test:\nW-Wert = {shapiro_stat:.3f}\n{p_text(shapiro_p)}', transform=ax.transAxes, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.8))
plt.tight_layout()
plt.show()

print(f'Geschätzte Parameter für {variable}: μ = {mu:.2f}, σ = {sigma:.2f}')
if shapiro_p > 0.05:
    print('✅ Kein Widerspruch zur Normalverteilung (p > 0.05). Das ist kein Beweis, bei nur 19 Werten ist der Test wenig aussagekräftig.')
else:
    print('⚠️  Die Daten passen schlecht zu einer Normalverteilung (p ≤ 0.05).')

**Wie interpretiert man das Ergebnis des Shapiro-Wilk-Tests? (optionale Vertiefung)**

Der Shapiro-Wilk-Test prüft, ob die Daten zu einer Normalverteilung passen.
- Die sogenannte *Nullhypothese* lautet: Die Daten sind normalverteilt.
- Der Test berechnet eine **Teststatistik** und einen **p-Wert**.

**Die Teststatistik (W-Wert):**
- Liegt zwischen 0 und 1
- W = 1 bedeutet: perfekt normalverteilt. Kleinere Werte bedeuten stärkere Abweichung.
- Ab welchem W eine Abweichung "zu groß" ist, hängt aber von der Anzahl der Daten ab. Bei sehr vielen Daten wird schon ein W knapp unter 1 abgelehnt, bei sehr wenigen Daten erst ein deutlich kleineres W.
- Deshalb rechnet der Test W in einen p-Wert um, und **nur der p-Wert entscheidet**.

**Der p-Wert:**
- Ist der p-Wert **größer als 0.05**, gibt es keinen Hinweis auf eine Abweichung von der Normalverteilung. Die Daten *könnten* also normalverteilt sein.
- Ist der p-Wert **kleiner oder gleich 0.05**, passen die Daten schlecht zu einer Normalverteilung. Wir **lehnen die Nullhypothese ab** ("normalverteilt").

**Achtung:**
- Das W alleine nicht interpretieren, immer den p-Wert ansehen.
- Auch wenn der p-Wert groß ist, heißt das nicht, dass die Daten garantiert normalverteilt sind – nur, dass man es mit den vorliegenden Daten nicht widerlegen kann.
- Bei kleinen Stichproben (hier nur 19 Personen) erkennt der Test Abweichungen schlecht. Ein großer p-Wert bedeutet dann wenig.
- Der p-Wert ist **nicht** die Wahrscheinlichkeit, dass die Daten normalverteilt sind (mehr dazu weiter unten).

**Zu den Zahlen im Plot:**
- Das σ in der Legende (Dichtekurve) ist etwas kleiner als `std` in `describe()`. Grund: `norm.fit` teilt durch *n*, pandas durch *n − 1*. Beide Varianten sind üblich.
- Wie viele Klassen (Bins) das Histogramm hat, beeinflusst das Bild stark. Bei 19 Werten sind 6–8 Klassen sinnvoll. Mit zu vielen Klassen entstehen leere Balken und das Histogramm wirkt zufällig.

**Tipp:**
Schau dir immer auch das Histogramm an! Test und Grafik zusammen geben das beste Bild.

## 3. Scatterplot mit linearem Fit

Wir untersuchen den Zusammenhang zwischen **Körpergröße** und **Gewicht**.

In [ ]:
var_x = 'Körpergröße [cm]'
var_y = 'Gewicht [kg]'
x_data = df[var_x].values
y_data = df[var_y].values
unit_x = var_x.split('[')[1][:-1]  # Einheit aus dem Spaltennamen, z.B. cm
unit_y = var_y.split('[')[1][:-1]  # z.B. kg

slope, intercept, r_value, p_value, std_err = stats.linregress(x_data, y_data)
r_squared = r_value**2

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(x_data, y_data, s=100, alpha=0.7, color='blue', edgecolors='black')
x_fit = np.linspace(x_data.min() - 2, x_data.max() + 2, 100)
y_fit = slope * x_fit + intercept
ax.plot(x_fit, y_fit, 'r-', linewidth=2, label=f'y = {slope:.2f}x + {intercept:.1f}')
ax.set_xlabel(var_x)
ax.set_ylabel(var_y)
ax.set_title(f'Scatterplot: {var_x} vs. {var_y}')
ax.legend()
ax.grid(True, alpha=0.3)
ax.text(0.02, 0.98, f'Lineare Regression:\nSteigung = {slope:.2f}\nR² = {r_squared:.3f}\nKorrelation r = {r_value:.3f}\n{p_text(p_value)}', transform=ax.transAxes, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.8))
plt.tight_layout()
plt.show()

print(f'Lineare Regression: {var_x} (x) und {var_y} (y)')
print(f'   Steigung: {slope:.3f} {unit_y} pro {unit_x}')
print(f'   Y-Achsenabschnitt: {intercept:.2f} {unit_y} (reine Rechengröße, siehe Erklärung unten)')
print(f'   Korrelationskoeffizient r: {r_value:.3f}')
print(f'   Bestimmtheitsmaß R²: {r_squared:.3f} (ca. {r_squared*100:.0f}% der Streuung im Gewicht lässt sich durch die Gerade beschreiben)')
signifikanz = 'signifikant' if p_value < 0.05 else 'nicht signifikant'
print(f'   {p_text(p_value)} ({signifikanz} bei α = 0.05)')

**Was bedeuten die Werte aus `stats.linregress`?**

- **slope**: Die Steigung der Regressionsgeraden. Sie gibt an, um wie viel sich die Zielvariable (y) im Mittel ändert, wenn die erklärende Variable (x) um 1 steigt.
- **intercept**: Der y-Achsenabschnitt der Regressionsgeraden, also der Wert der Geraden bei x = 0. Hier wäre das das "Gewicht" bei 0 cm Körpergröße. Das ist nur eine **Rechengröße** und nicht sinnvoll interpretierbar. Die Gerade gilt nur im Bereich der Daten (hier ca. 160-188 cm).
- **r_value**: Der Korrelationskoeffizient (r, Pearson). Er misst die Stärke und Richtung des **linearen** Zusammenhangs zwischen x und y. Werte nahe +1 oder -1 bedeuten einen starken linearen Zusammenhang, Werte nahe 0 *keinen linearen* Zusammenhang. Ein Zusammenhang, der nicht geradlinig ist (z.B. eine U-Kurve), kann trotzdem r ≈ 0 haben.
- **p_value**: Der p-Wert zum Test, ob die Steigung (slope) signifikant von 0 verschieden ist.
    - **Achtung:** Dieser p-Wert prüft NICHT auf Normalverteilung, sondern ob ein linearer Zusammenhang zwischen x und y statistisch nachweisbar ist.
    - **α (Alpha)**: Das Signifikanzniveau ist ein fester Schwellenwert (meist 0.05 = 5%), den wir vorher festlegen. Der p-Wert wird mit diesem Alpha verglichen, um zu entscheiden, ob das Ergebnis statistisch signifikant ist.
    - "Signifikant" heißt **nicht** "wichtig" oder "stark". Bei sehr vielen Daten wird auch ein winziger Zusammenhang signifikant.
- **std_err**: Der Standardfehler der Steigung. Er gibt an, wie stark die geschätzte Steigung in verschiedenen Stichproben schwanken würde.

**Zusammengefasst:**
- Der p-Wert aus `linregress` ist ein Maß für die Signifikanz des linearen Zusammenhangs, nicht für Normalverteilung!
- Der p-Wert aus dem Shapiro-Wilk-Test prüft auf Normalverteilung.
- α (Alpha) ist das Signifikanzniveau - ein Schwellenwert, mit dem der p-Wert verglichen wird.

**Zwei häufige Verwechslungen:**
- Der Shapiro-Wilk-Test oben prüft nur die *Körpergrößen*. Für die Regression sind nicht die Größen selbst entscheidend, sondern die Abweichungen der Punkte von der Geraden (die *Residuen*). Das vertiefen wir später.
- Die Regression zeigt einen **Zusammenhang**, aber keine **Ursache**. Dass Körpergröße und Gewicht zusammenhängen, heißt nicht, dass das eine das andere verursacht (siehe Abschnitt 4).

**Was bedeutet der p-Wert genau? (optionale Vertiefung)**

Der p-Wert beantwortet die Frage: *Wenn die Nullhypothese stimmt, wie wahrscheinlich wären dann Daten, die mindestens so stark von ihr abweichen wie unsere?*

**Bei der linearen Regression:**
- **Nullhypothese:** Es gibt keinen linearen Zusammenhang (Steigung = 0)
- **p = 0.01 bedeutet:** Wenn es in Wahrheit keinen linearen Zusammenhang gäbe, würde man nur in 1% der Stichproben eine Steigung beobachten, die mindestens so weit von 0 entfernt ist (nach oben oder unten)
- **Interpretation:** Da 0.01 < 0.05 → Nullhypothese wird abgelehnt, ein linearer Zusammenhang ist statistisch nachweisbar

**Bei Shapiro-Wilk:**
- **Nullhypothese:** Die Daten sind normalverteilt
- **p = 0.01 bedeutet:** Wenn die Daten wirklich normalverteilt wären, würde man nur in 1% der Stichproben eine mindestens so starke Abweichung von der Normalverteilung sehen
- **Interpretation:** Da 0.01 < 0.05 → Nullhypothese wird abgelehnt, die Daten passen schlecht zu einer Normalverteilung

**Wichtig:** Der p-Wert ist NICHT die Wahrscheinlichkeit, dass die Nullhypothese wahr ist, und auch nicht die Wahrscheinlichkeit, dass das Ergebnis "nur Zufall" ist.

**Was bedeutet ein kleiner p-Wert bei verschiedenen Tests?**

| Test | Nullhypothese H₀ | Kleiner p-Wert (≤ 0.05) | Großer p-Wert (> 0.05) |
|---|---|---|---|
| Shapiro-Wilk | Die Daten sind normalverteilt | H₀ wird abgelehnt: Daten passen schlecht zur Normalverteilung | Kein Widerspruch zur Normalverteilung (kein Beweis!) |
| Regression (Steigung) | Steigung = 0 (kein linearer Zusammenhang) | H₀ wird abgelehnt: linearer Zusammenhang nachweisbar | Kein Nachweis eines linearen Zusammenhangs (kein Beweis für "kein Zusammenhang") |

Bei **beiden** Tests gilt dieselbe Regel: Ein kleiner p-Wert (< 0.05) führt zum **Ablehnen der Nullhypothese**.

**Warum sehen die Nullhypothesen "umgekehrt" aus?** Sie sind es nicht. H₀ ist immer die "langweilige" Ausgangsannahme: *nichts Auffälliges, kein Effekt*. Ein Test kann H₀ nur widerlegen, aber nie bestätigen.
- Bei Shapiro-Wilk ist "nichts Auffälliges" die Normalverteilung. Ein kleiner p-Wert bedeutet daher: **Abweichung von der Normalverteilung gefunden**. Das ist meist das unerwünschte Ergebnis, wenn man eine Normalverteilung voraussetzen möchte.
- Bei der Regression ist "nichts Auffälliges" der fehlende Zusammenhang. Ein kleiner p-Wert bedeutet daher: **Zusammenhang gefunden**. Das ist meist das gewünschte Ergebnis.

Deshalb heißt "p < 0.05" bei der Regression oft "gut" und bei Shapiro-Wilk oft "schlecht". Lies immer zuerst nach, was die Nullhypothese des jeweiligen Tests besagt!

## 4. Korrelationskoeffizient mit Python-Funktion

Berechnung der Korrelation zwischen **Körpergröße** und **Schuhgröße** mit `numpy`, `scipy` und `pandas`.

Es gibt zwei gängige Korrelationskoeffizienten:
- **Pearson (r)**: misst den *linearen* Zusammenhang der Werte. `np.corrcoef`, `stats.pearsonr` und `df.corr()` (Standardeinstellung) berechnen alle Pearson.
- **Spearman (ρ)**: rechnet nicht mit den Werten, sondern mit deren **Rangplätzen** (kleinster Wert = Rang 1 usw.). Er erkennt auch Zusammenhänge, die gleichmäßig steigen oder fallen, aber nicht geradlinig sind, und ist weniger empfindlich gegen Ausreißer.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['Körpergröße [cm]'], df['Schuhgröße [EU]'], s=80, alpha=0.7, edgecolors='black')
ax.set_xlabel('Körpergröße [cm]')
ax.set_ylabel('Schuhgröße [EU]')
ax.set_title('Körpergröße und Schuhgröße')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
var1 = 'Körpergröße [cm]'
var2 = 'Schuhgröße [EU]'
data1 = df[var1].values
data2 = df[var2].values

numpy_corr = np.corrcoef(data1, data2)[0, 1]
scipy_corr, scipy_p = stats.pearsonr(data1, data2)
pandas_corr = df[[var1, var2]].corr().iloc[0, 1]
spearman_corr, spearman_p = stats.spearmanr(data1, data2)

print(f'Korrelation zwischen {var1} und {var2}:')
print(f'   NumPy corrcoef():     r = {numpy_corr:.4f}')
print(f'   SciPy pearsonr():     r = {scipy_corr:.4f}, {p_text(scipy_p)}')
print(f'   Pandas corr():        r = {pandas_corr:.4f}')
print(f'   Spearman-Korrelation: ρ = {spearman_corr:.4f}, {p_text(spearman_p)}')

**Wie interpretiert man das Ergebnis?**

Grobe Faustregel für den Betrag |r| (je nach Fachgebiet unterschiedlich):
- unter 0.3: schwacher linearer Zusammenhang
- 0.3 bis 0.7: mittlerer linearer Zusammenhang
- über 0.7: starker linearer Zusammenhang

Hier: Körpergröße und Schuhgröße haben r ≈ 0.80, also einen **starken positiven** Zusammenhang: Größere Personen haben im Mittel größere Schuhe. Zum Vergleich: Körpergröße und Gewicht (Abschnitt 3) haben nur r ≈ 0.57.

**Korrelation ist nicht Kausalität!**
Große Schuhe machen niemanden größer, und mehr Gewicht macht niemanden größer. Solche Zusammenhänge entstehen oft durch gemeinsame Einflüsse (z.B. Geschlecht oder Alter), die in den Daten nicht vorkommen. Eine Korrelation zeigt nur, dass zwei Merkmale zusammen auftreten, nicht warum.

**Pearson und Spearman unterscheiden sich hier deutlich** (r ≈ 0.80, ρ ≈ 0.56). Grund: Pearson wird stark von den beiden kleinsten Personen (160 und 167 cm, kleine Schuhe) beeinflusst. Lässt man sie weg, sinkt r auf etwa 0.47. Spearman nutzt nur die Rangplätze, dort zählt jede Person gleich viel. Merke: Einzelne Extremwerte können eine Korrelation stark beeinflussen. Schau dir deshalb immer auch den Scatterplot an!

## 5. Korrelationskoeffizient explizit berechnen

Schritt-für-Schritt Berechnung nach der Definition: $r = \frac{\sum{(x_i - \bar{x})(y_i - \bar{y})}}{\sqrt{\sum{(x_i - \bar{x})^2} \sum{(y_i - \bar{y})^2}}}$

In [ ]:
x = data1
y = data2

# 1. Mittelwerte
x_mean = np.mean(x)
y_mean = np.mean(y)

# 2. Abweichungen vom Mittelwert
x_diff = x - x_mean
y_diff = y - y_mean

# 3. Zähler: Summe der Produkte der Abweichungen
sum_xy_products = np.sum(x_diff * y_diff)

# 4. Nenner: Wurzel aus (Summe der quadrierten x-Abweichungen * Summe der quadrierten y-Abweichungen)
sum_x_squared = np.sum(x_diff**2)
sum_y_squared = np.sum(y_diff**2)
denominator = np.sqrt(sum_x_squared * sum_y_squared)

r_manual = sum_xy_products / denominator
print('Formel: r = Σ(xi - x̄)(yi - ȳ) / √[Σ(xi - x̄)² × Σ(yi - ȳ)²]')
print(f'Manual: r = {r_manual:.4f}')
print(f'NumPy:  r = {numpy_corr:.4f}')
print('Gleiches Ergebnis:', np.isclose(r_manual, numpy_corr))